# AIC Model Comparison and Direct REML Fitting

This tutorial covers two features of the PGAM library that go beyond the basic fit:

1. **Corrected AIC** — a bias-corrected information criterion for comparing models with different covariate sets.
2. **Direct REML** — an alternative fitting strategy that estimates smoothing parameters by marginal likelihood rather than cross-validation.

Both features build on the same synthetic Poisson dataset used in the main tutorial (spatial tuning + event kernel + uninformative nuisance variable).

In [8]:
import numpy as np
import statsmodels.api as sm
import scipy.stats as sts
import matplotlib.pyplot as plt

from PGAM.GAM_library import *
import PGAM.gam_data_handlers as gdh

%matplotlib inline

In [9]:
# --------------------------------------------------------------------------
# Synthetic dataset (identical to the main tutorial)
# --------------------------------------------------------------------------
rng = np.random.default_rng(42)

time_points = 50_000          # ~5 min at 6 ms bins
rate        = 5.0 * 0.006     # target mean rate (spikes / bin)
trial_ids   = np.repeat(np.arange(50), time_points // 50)

# Spatial variable (drives the neuron)
spatial_var  = rng.normal(scale=np.sqrt(5), size=time_points)
spatial_var  = np.clip(spatial_var, -5, 5)

# Nuisance variable (does NOT drive the neuron)
nuisance_var = rng.normal(scale=np.sqrt(5), size=time_points)
nuisance_var = np.clip(nuisance_var, -5, 5)

# Temporal event (drives the neuron via a causal kernel)
events = np.zeros(time_points)
events[rng.choice(time_points, 600, replace=False)] = 1

# True spatial response (half-bell shape)
knots_true  = np.hstack(([-5]*3, np.linspace(-5, 5, 8), [5]*3))
beta_true   = np.array([0, 0.2, 0.5, 1.0, 1.2, 0.9, 0.4, 0.1, 0, 0])
resp_spatial = gdh.splineDesign(knots_true, spatial_var, 4, der=0) @ beta_true

# True temporal kernel (gamma-shaped)
kernel = sts.gamma.pdf(np.linspace(0, 20, 100), a=2) * 2
kernel = np.hstack((np.zeros(101), kernel))
resp_temporal = np.zeros(time_points)
for tr in np.unique(trial_ids):
    mask = trial_ids == tr
    resp_temporal[mask] = np.convolve(events[mask], kernel, mode='same')

log_mu = resp_spatial + resp_temporal
log_mu -= np.log(np.mean(np.exp(log_mu)) / rate)
spk_counts = rng.poisson(np.exp(log_mu))

print(f'Mean rate: {spk_counts.mean() / 0.006:.1f} Hz,  non-zero bins: {(spk_counts > 0).mean():.1%}')

Mean rate: 4.8 Hz,  non-zero bins: 2.9%


In [10]:
from PGAM.der_wrt_smoothing import d2variance_family, deriv3_link

# Build the smooths_handler (spatial + nuisance + temporal)
knots = np.hstack(([-5]*3, np.linspace(-5, 5, 8), [5]*3))

base = sm.families.Poisson(link=sm.families.links.Log())

# wrap link and family to get custom classes with extra properties
# like higher order derivatives needed to compute the AIC correction
base.link = deriv3_link(base.link, run_tests=False)
fam = d2variance_family(base, run_tests=False)

sm_handler = smooths_handler()
sm_handler.add_smooth('spatial',  [spatial_var],  knots=[knots], ord=4,
                      is_temporal_kernel=False, trial_idx=trial_ids,
                      is_cyclic=[False], penalty_type='der', der=2)
sm_handler.add_smooth('nuisance', [nuisance_var], knots=[knots], ord=4,
                      is_temporal_kernel=False, trial_idx=trial_ids,
                      is_cyclic=[False], penalty_type='der', der=2)
sm_handler.add_smooth('temporal', [events], knots=None, ord=4,
                      is_temporal_kernel=True, trial_idx=trial_ids,
                      penalty_type='der', der=2, knots_num=8,
                      kernel_length=201, kernel_direction=1)

train_mask = trial_ids % 5 != 0   # 80 % of trials for training

pgam = general_additive_model(
    sm_handler, sm_handler.smooths_var, spk_counts, fam
)

---

## 1. Corrected AIC

### Why standard AIC is biased for GAMs

The classical AIC penalises model complexity by the number of free parameters $k$:

$$\text{AIC} = -2\,\ell(\hat{\beta}) + 2k$$

In a GLM with no regularisation $k = p$ (one parameter per coefficient). In a GAM the smoothing penalties shrink many directions of $\beta$ toward zero, so the effective model complexity is much lower than $p$. Using $k = p$ would unfairly penalise smooth models.

### The formula (Wood 2017, eq. 6.32)

A brief reminder of notation: $\beta$ are the B-spline coefficients (one vector per covariate, concatenated); $\lambda$ are the smoothing parameters that control how wiggly each tuning function is allowed to be (one per covariate, estimated from data); $\hat{\beta}$ and $\hat{\lambda}$ are their estimated values.

The corrected AIC replaces $2k$ with $2\tau_2$:

$$\text{AIC} = -2\,\ell(\hat{\beta}) + 2\tau_2, \qquad \tau_2 = \operatorname{tr}(\mathbf{V}'_\beta\, \mathcal{I})$$

where $\ell$ is the *unpenalised* log-likelihood at $\hat{\beta}$ and $\mathcal{I} = -\partial^2\ell/\partial\beta^2$ is the observed Fisher information. $\mathbf{V}'_\beta$ is the **Bayesian covariance matrix of $\beta$ corrected for smoothing parameter uncertainty**:

$$\mathbf{V}'_\beta = \underbrace{(\mathcal{I} + S_\lambda)^{-1}}_{\mathbf{V}_\beta} + \underbrace{\mathbf{J}\mathbf{V}_\rho\mathbf{J}^\top + \text{(Cholesky correction)}}_{\text{smoothing uncertainty}}$$

- $\mathbf{V}_\beta = (\mathcal{I} + S_\lambda)^{-1}$ is the posterior covariance of $\beta$ treating $\hat{\lambda}$ as fixed and known.
- The remaining terms account for the fact that $\hat{\lambda}$ is itself estimated from data and therefore uncertain. They propagate that uncertainty into $\hat{\beta}$ via $\mathbf{J} = \mathrm{d}\hat{\beta}/\mathrm{d}\rho$ (how much the spline coefficients shift when the smoothing level changes, $\rho = \log\lambda$) and $\mathbf{V}_\rho$ (how uncertain the smoothing level estimate is, given by the inverse Hessian of the REML objective). This is the Kass & Steffey (1989) correction extended to GAMs by Wood et al. (2016).

Ignoring the smoothing uncertainty terms causes the AIC to systematically favour larger models. Wood et al. (2016) show via simulation that eq. (6.32) substantially reduces this bias.

> **Reference:** Wood, S.N. (2017). *Generalized Additive Models: An Introduction with R*, 2nd ed., CRC Press, §6.11.2; Wood et al. (2016), *JASA*.

### How to enable AIC

Pass `compute_AIC=True` to `fit_full_and_reduced`. The result is stored in `result.AIC`. It works with any fitting approach (`pql_gcv`, `direct_reml`) and adds only a small overhead — one additional covariance matrix computation at the end of fitting.

In [15]:
full_gcv, reduced_gcv = pgam.fit_full_and_reduced(
    sm_handler.smooths_var,
    th_pval=0.001,
    max_iter=100,
    use_dgcv=3., # default `use_dgcv= True` sets the gcv scaling to 1.5,
                # smaller problems may require more smoothing, one could set that by
                # passing a value, the larger the value the smoother the fit
    trial_num_vec=trial_ids,
    filter_trials=train_mask,
    compute_AIC=True,            # ← enable corrected AIC
)

print('Full model')
print(f'  variables : {list(full_gcv.var_list)}')
print(f'  AIC       : {full_gcv.AIC:.2f}')
print()
print('Reduced model (statistically selected variables only)')
print(f'  variables : {list(reduced_gcv.var_list)}')
print(f'  AIC       : {reduced_gcv.AIC:.2f}')
print()
print(f'ΔAIC (full − reduced) = {full_gcv.AIC - reduced_gcv.AIC:.2f}')
print('Negative means the reduced model is preferred (less complex, similar fit).')

Full model
  variables : ['spatial', 'nuisance', 'temporal']
  AIC       : 10368.61

Reduced model (statistically selected variables only)
  variables : [np.str_('spatial'), np.str_('temporal')]
  AIC       : 10367.08

ΔAIC (full − reduced) = 1.53
Negative means the reduced model is preferred (less complex, similar fit).


### Manual model comparison

You can also compare hand-picked model variants. Here we fit two models:
- **Full**: spatial + nuisance + temporal  
- **No-nuisance**: spatial + temporal only

The AIC difference quantifies whether the nuisance variable contributed beyond chance.

In [12]:
# Fit a model that excludes the nuisance variable from the start
sm_no_nuis = smooths_handler()
sm_no_nuis.add_smooth('spatial',  [spatial_var], knots=[knots], ord=4,
                      is_temporal_kernel=False, trial_idx=trial_ids,
                      is_cyclic=[False], penalty_type='der', der=2)
sm_no_nuis.add_smooth('temporal', [events], knots=None, ord=4,
                      is_temporal_kernel=True, trial_idx=trial_ids,
                      penalty_type='der', der=2, knots_num=8,
                      kernel_length=201, kernel_direction=1)

pgam_no_nuis = general_additive_model(
    sm_no_nuis, sm_no_nuis.smooths_var, spk_counts, fam
)
fit_no_nuis, _ = pgam_no_nuis.fit_full_and_reduced(
    sm_no_nuis.smooths_var,
    th_pval=0.001, max_iter=100, use_dgcv=True,
    trial_num_vec=trial_ids, filter_trials=train_mask,
    compute_AIC=True,
)

print('Model comparison via corrected AIC')
print(f'  Full (spatial + nuisance + temporal) : {full_gcv.AIC:.2f}')
print(f'  No nuisance (spatial + temporal)     : {fit_no_nuis.AIC:.2f}')
delta = full_gcv.AIC - fit_no_nuis.AIC
verdict = 'nuisance adds nothing' if delta > 0 else 'nuisance helps'
print(f'  ΔAIC = {delta:.2f}  → {verdict}')

Model comparison via corrected AIC
  Full (spatial + nuisance + temporal) : 10362.68
  No nuisance (spatial + temporal)     : 10355.78
  ΔAIC = 6.90  → nuisance adds nothing


**Interpreting ΔAIC.** A rule of thumb from information theory:

| ΔAIC (worse − better) | Interpretation |
|---|---|
| < 2 | Indistinguishable |
| 2 – 6 | Moderate evidence |
| > 10 | Strong evidence |

Because the AIC uses $\tau_2 = \operatorname{tr}(\mathbf{V}'_\beta \mathcal{I})$ rather than the raw parameter count, a heavily regularised (near-zero) nuisance smooth contributes very little to the penalty term — the model is not punished as if it had $p_{\text{nuisance}}$ free parameters.

---

## 2. Direct REML fitting

### How the default (GCV) fitting works

The default PGAM fitting strategy (**PQL-GCV**) alternates two steps:

1. **Inner loop (PIRLS):** given the current smoothing parameters $\lambda$, find the penalised MLE $\hat{\beta}(\lambda)$ via iteratively reweighted least squares, warm-started from the previous iterate.
2. **Outer loop:** update $\lambda$ to minimise the GCV score, reusing the matrix factorisations already computed in the inner loop.

### What REML does differently

**Restricted Maximum Likelihood (REML)** treats the B-spline coefficients $\beta$ as random effects and marginalises them out analytically (via the Laplace approximation), yielding a marginal likelihood over $\rho = \log\lambda$ only:

$$\text{REML}(\rho) \approx \ell(\hat{\beta}) - \frac{1}{2}\log|H + S_\lambda| + \frac{1}{2}\log|S_\lambda|_+$$

where $H = -\partial^2\ell/\partial\beta^2$ is the Fisher information at the mode. Maximising REML over $\rho$ gives a principled, Bayesian-flavoured smoothing estimate — the same criterion used in `mgcv::gam(..., method="REML")` in R.

### Why it is more expensive

Each evaluation of the REML objective requires a **full inner BFGS optimisation** to find $\hat{\beta}(\rho)$, without the warm-starting that makes PIRLS efficient. In practice Direct REML is noticeably slower than PQL-GCV, especially for models with many covariates. Use it for **final fits on a confirmed, small covariate set** — not for exploratory screening.

> **Reference:** Wood, S.N. (2017). *Generalized Additive Models: An Introduction with R*, 2nd ed., CRC Press, §6.6.

### PQL-REML: fast linearized REML (recommended)

**PQL-REML** is the middle ground between GCV and Direct REML.  It keeps the efficient PIRLS loop of GCV but replaces the GCV score with a REML criterion evaluated on the *linearised* working model at each PIRLS step.

At iteration $t$, the Fisher-scoring linearisation produces a working Gaussian model
$$y' = \sqrt{W}\,z, \qquad X' = \sqrt{W}\,X$$
(where $W$ are the IRLS weights and $z$ is the adjusted dependent variable).  PQL-REML then selects $\rho = \log\lambda$ by minimising the **negative restricted log-likelihood of this working model**:

$$-\ell_r(\rho) = \tfrac{1}{2}\bigl[\|y'\|^2 - \|U_1^\top Q^\top y'\|^2\bigr] + \tfrac{1}{2}\log|X'^\top X' + S_\lambda| - \tfrac{1}{2}\log|S_\lambda|_+$$

where $Q R$ is the QR factorisation of $X'$ and $U_1$ comes from the SVD of $[R;\,B_\lambda]$ ($B_\lambda$ is the penalty square root).  The first bracketed term is the **REML RSS** $y'^\top(I-A)y'$ — *not* $\|y'-Ay'\|^2$, because the hat matrix $A$ is not idempotent for penalised regression.

**Why use PQL-REML over GCV?**

| | GCV | PQL-REML | Direct REML |
|---|---|---|---|
| Speed | fast | fast | slow |
| Score | cross-validation proxy (asympt. MSE-optimal) | linearized REML | full REML |
| Undersmoothing risk | moderate | low | low |
| Recommended use | exploration / screening | **standard fits** | final / small models |

REML-based criteria are less prone to the undersmoothing bias that GCV can exhibit with correlated errors.  PQL-REML achieves this without the cost of a full inner optimisation at every objective evaluation.

Pass `fitting_approach="pql_reml"` to use it.

> **Reference:** Wood, S.N. (2017). *Generalized Additive Models: An Introduction with R*, 2nd ed., §6.5 (strategy 2).

In [ ]:
full_reml_pql, reduced_reml_pql = pgam.fit_full_and_reduced(
    sm_handler.smooths_var,
    th_pval=0.001,
    max_iter=100,
    use_dgcv=True,
    trial_num_vec=trial_ids,
    filter_trials=train_mask,
    fitting_approach="pql_reml",   # ← key flag
    conv_criteria="reml",          # monitor REML score for convergence
    compute_AIC=True,
)

print('PQL-REML')
print(f'  Selected variables : {list(reduced_reml_pql.var_list)}')
print(f'  Smoothing params   : {full_reml_pql.smooth_pen.round(3)}')
print(f'  AIC (full model)   : {full_reml_pql.AIC:.2f}')

### Direct REML with L-BFGS-B

L-BFGS-B approximates the Hessian of the REML objective from the history of gradient steps (quasi-Newton). It supports box constraints on $\rho$, which prevent the smoothing parameters from collapsing to zero or diverging — both pathological cases that can occur with an uninformative covariate.

Pass `fitting_approach="direct_reml"` to any call to `fit_full_and_reduced`.

In [13]:
full_reml_lbfgsb, reduced_reml_lbfgsb = pgam.fit_full_and_reduced(
    sm_handler.smooths_var,
    th_pval=0.001,
    max_iter=100,
    use_dgcv=True,
    trial_num_vec=trial_ids,
    filter_trials=train_mask,
    fitting_approach="direct_reml",  # ← key flag
    method="L-BFGS-B",               # outer optimiser (default for direct_reml)
    compute_AIC=True,
    bounds_rho=[(-10, 10)]
)

print('Direct REML (L-BFGS-B)')
print(f'  Selected variables : {list(reduced_reml_lbfgsb.var_list)}')
print(f'  Smoothing params   : {full_reml_lbfgsb.smooth_pen.round(3)}')
print(f'  AIC (full model)   : {full_reml_lbfgsb.AIC:.2f}')

Direct REML (L-BFGS-B)
  Selected variables : [np.str_('spatial'), np.str_('temporal')]
  Smoothing params   : [1.4274000e+01 9.2600000e-01 2.2026466e+04 2.2026466e+04 1.2094200e+02
 1.9850000e+00]
  AIC (full model)   : 10369.86


### Comparing GCV, PQL-REML, and Direct REML estimates

In [ ]:
x_grid  = np.linspace(-5, 5, 200)
var_names = list(full_gcv.var_list)

fig, axes = plt.subplots(1, len(var_names), figsize=(5 * len(var_names), 4))
if len(var_names) == 1:
    axes = [axes]

for ax, var in zip(axes, var_names):
    ax.set_title(var)
    for model, label, color in [
        (full_gcv,           'GCV',              'steelblue'),
        (full_reml_pql,      'PQL-REML',         'seagreen'),
        (full_reml_lbfgsb,   'Direct REML',      'darkorange'),
    ]:
        if var not in model.var_list:
            continue
        if var == 'temporal':
            dim_kern = model.smooth_info[var]['basis_kernel'].shape[0]
            impulse  = np.zeros(dim_kern)
            impulse[(dim_kern - 1) // 2] = 1
            x_in = impulse
            xplot = (np.arange(dim_kern) - (dim_kern - 1) // 2) * 0.006
        else:
            x_in = x_grid
            xplot = x_grid

        mean_log, lo_log, hi_log = model.smooth_compute([x_in], var)
        ax.plot(xplot, mean_log, color=color, label=label)
        ax.fill_between(xplot, lo_log, hi_log, color=color, alpha=0.2)

    ax.legend(fontsize=9)
    ax.set_xlabel('x' if var != 'temporal' else 'time after event [s]')
    ax.set_ylabel('log firing rate (centered)')

plt.tight_layout()
plt.suptitle('GCV vs PQL-REML vs Direct REML smoothing estimates', y=1.02)
plt.show()

### Practical guidance

**Recommended workflow:**

1. Screen all candidate covariates with the default PQL-GCV fit to identify the minimal subset (use `fit_full_and_reduced`). This is fast enough to run on every recorded neuron.
2. Re-fit the confirmed covariate set with **PQL-REML** (`fitting_approach="pql_reml"`, `conv_criteria="reml"`). It is as fast as GCV but uses a statistically sounder criterion.
3. Use **Direct REML** (`fitting_approach="direct_reml"`) when you want the most principled estimate on a small, fixed covariate set and can afford the extra runtime.
4. Use `compute_AIC=True` in all steps — the corrected AIC gives a complementary model-selection signal to the p-value-based variable selection.
5. Pass `bounds_rho` with Direct REML if the smoothing level diverges for weakly informative covariates.